In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_unixtime, to_timestamp, row_number, dense_rank,
    to_date, year, month, dayofmonth, desc
)
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

bronze_table = "dev.bronze.batch_item_properties"
silver_table = "dev.silver.batch_silver"

# Read Bronze data
bronze_df = spark.sql(f"SELECT * FROM {bronze_table}")
print(f"📊 Bronze records: {bronze_df.count()}")
bronze_df.display()

## 2. Data Cleaning & Type Conversion

In [ ]:
# Convert timestamp and add date columns
cleaned_df = bronze_df.select(
    col("timestamp").cast("long").alias("timestamp_unix"),
    from_unixtime(col("timestamp") / 1000, "yyyy-MM-dd HH:mm:ss").cast("timestamp").alias("event_time"),
    col("itemid").cast("integer").alias("item_id"),
    col("property").cast("integer").alias("property_id"),
    col("value").cast("string").alias("property_value"),
    col("_ingestion_time").alias("ingestion_timestamp"),
    col("_source_file").alias("source_file")
).filter(col("item_id").isNotNull() & col("property_id").isNotNull())

# Add date partitioning columns
cleaned_df = cleaned_df.select(
    "*",
    to_date(col("event_time")).alias("event_date"),
    year(col("event_time")).alias("event_year"),
    month(col("event_time")).alias("event_month")
)

print(f"✅ Cleaned records: {cleaned_df.count()}")
cleaned_df.display()

## 3. Deduplication (Most Recent Record per Property)

In [ ]:
# Keep only latest value for each item-property combination
window_spec = Window.partitionBy("item_id", "property_id").orderBy(desc("timestamp_unix"))

deduped_df = cleaned_df.select(
    "*",
    row_number().over(window_spec).alias("rn")
).filter(col("rn") == 1).drop("rn")

print(f"✅ Deduplicated records: {deduped_df.count()}")
print(f"Deduplication ratio: {cleaned_df.count() / deduped_df.count():.2f}x")

## 4. Write to Silver Table (Partitioned)

In [ ]:
# Write to Silver with partitioning by event_date
(
    deduped_df
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("event_year", "event_month", "event_date")
    .option("mergeSchema", "true")
    .saveAsTable(silver_table)
)

print(f"✅ Data written to {silver_table}")

## 5. Validate Silver Table

In [ ]:
# Check partition structure
spark.sql(f"DESCRIBE FORMATTED {silver_table}").display()

# Sample data
spark.sql(f"SELECT * FROM {silver_table} LIMIT 20").display()

## 6. Quality Metrics

In [ ]:
# Quality stats
spark.sql(f"""
SELECT 
  COUNT(*) as total_records,
  COUNT(DISTINCT item_id) as unique_items,
  COUNT(DISTINCT property_id) as unique_properties,
  MIN(event_time) as earliest_event,
  MAX(event_time) as latest_event,
  COUNT(DISTINCT event_date) as date_range
FROM {silver_table}
""").display()